In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [2]:
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer

In [45]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

In [3]:
# Load data
data_folder = Path("Dataset/")
train_data = pd.read_csv(data_folder/"Train.csv")
test_data = pd.read_csv(data_folder/ "Test.csv")
submission_data = pd.read_csv(data_folder/"Submission.csv")

In [4]:
train_data.head()

,pickup_datetime,dropoff_datetime,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,passenger_count,traffic_congestion_level,pickup_borough,dropoff_borough,tolls_amount,tip_amount,fare_estimate,trip_distance_miles
0,3/22/24 9:10,3/22/24 9:23,40.690045,-74.239699,40.704018,-74.214368,2,high,Other,Other,0.00,0.14,18.40,10.9422
1,8/14/24 10:48,8/14/24 10:52,40.763224,-74.053604,40.759277,-74.066002,1,medium,Other,Other,0.00,1.36,9.99,1.2817
2,6/17/24 3:47,6/17/24 3:55,40.658970,-74.179122,40.628631,-74.180385,1,low,Other,Staten Island,5.67,2.36,20.22,13.8034
3,6/25/24 0:45,6/25/24 1:04,40.696980,-73.773970,40.670444,-73.838194,2,medium,Brooklyn,Brooklyn,0.00,3.80,29.73,24.2266
4,6/10/24 4:10,6/10/24 4:14,40.580638,-73.788101,40.594286,-73.780625,3,low,Brooklyn,Brooklyn,0.00,2.68,11.05,3.4222


In [5]:
train_data.shape

(14699, 14)

In [6]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14699 entries, 0 to 14698
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   pickup_datetime           14699 non-null  object 
 1   dropoff_datetime          14699 non-null  object 
 2   pickup_latitude           14699 non-null  float64
 3   pickup_longitude          14699 non-null  float64
 4   dropoff_latitude          14699 non-null  float64
 5   dropoff_longitude         14699 non-null  float64
 6   passenger_count           14699 non-null  int64  
 7   traffic_congestion_level  14699 non-null  object 
 8   pickup_borough            14699 non-null  object 
 9   dropoff_borough           14699 non-null  object 
 10  tolls_amount              14699 non-null  float64
 11  tip_amount                14699 non-null  float64
 12  fare_estimate             14699 non-null  float64
 13  trip_distance_miles       14699 non-null  float64
dtypes: flo

In [7]:
train_data.isnull().sum().sum()

np.int64(0)

In [8]:
test_data.drop("trip_distance_miles",axis=1).isnull().sum()

pickup_datetime             0
dropoff_datetime            0
pickup_latitude             0
pickup_longitude            0
dropoff_latitude            0
dropoff_longitude           0
passenger_count             0
traffic_congestion_level    0
pickup_borough              0
dropoff_borough             0
tolls_amount                0
tip_amount                  0
fare_estimate               0
dtype: int64

In [9]:
train_data.describe()

,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,passenger_count,tolls_amount,tip_amount,fare_estimate,trip_distance_miles
count,14699.000000,14699.000000,14699.000000,14699.000000,14699.000000,14699.000000,14699.000000,14699.000000,14699.000000
mean,40.734509,-73.974803,40.735102,-73.974698,1.754201,1.156073,3.629243,23.465897,13.975935
std,0.106536,0.158391,0.110456,0.163223,1.025691,3.754118,2.878294,13.905790,7.900243
min,40.550003,-74.249989,40.450000,-74.350000,1.000000,0.000000,0.000000,4.660000,0.301800
25%,40.642862,-74.109936,40.643331,-74.110204,1.000000,0.000000,1.760000,14.070000,7.151200
50%,40.733822,-73.974071,40.733885,-73.974326,1.000000,0.000000,2.920000,19.780000,13.975000
75%,40.825650,-73.837649,40.826349,-73.837879,2.000000,0.000000,4.710000,28.690000,20.813350
max,40.919960,-73.700014,41.020000,-73.600000,6.000000,19.000000,30.810000,192.620000,27.670000


In [10]:
# convert datetime features into datatime data type
train_data["pickup_datetime"] = pd.to_datetime(train_data["pickup_datetime"],format="%m/%d/%y %H:%M")
train_data["dropoff_datetime"] = pd.to_datetime(train_data["dropoff_datetime"], format= "%m/%d/%y %H:%M")

test_data["pickup_datetime"] = pd.to_datetime(test_data["pickup_datetime"],format="%m/%d/%y %H:%M")
test_data["dropoff_datetime"] = pd.to_datetime(test_data["dropoff_datetime"], format= "%m/%d/%y %H:%M")

In [11]:
train_data[["pickup_datetime","dropoff_datetime"]].dtypes, test_data[["pickup_datetime","dropoff_datetime"]].dtypes

(pickup_datetime     datetime64[ns]
 dropoff_datetime    datetime64[ns]
 dtype: object,
 pickup_datetime     datetime64[ns]
 dropoff_datetime    datetime64[ns]
 dtype: object)

In [12]:
train_data.select_dtypes(exclude=object).corr()["trip_distance_miles"].sort_values(ascending=False)

trip_distance_miles    1.000000
fare_estimate          0.794198
tip_amount             0.598798
tolls_amount           0.171106
dropoff_datetime       0.006816
pickup_datetime        0.006755
passenger_count        0.003636
dropoff_latitude      -0.001130
pickup_longitude      -0.006374
pickup_latitude       -0.006858
dropoff_longitude     -0.007142
Name: trip_distance_miles, dtype: float64

In [13]:
new_date_time_train_df = pd.DataFrame({"month": train_data['pickup_datetime'].dt.month,
              "hour": train_data['pickup_datetime'].dt.hour,
              "week": train_data['pickup_datetime'].dt.isocalendar().week,
              "dayofweek": train_data['pickup_datetime'].dt.dayofweek
             })
# weekday feature
new_date_time_train_df["is_weekday"] = (new_date_time_train_df["dayofweek"]<=4).astype(int)
new_date_time_train_df.head()

,month,hour,week,dayofweek,is_weekday
0,3,9,12,4,1
1,8,10,33,2,1
2,6,3,25,0,1
3,6,0,26,1,1
4,6,4,24,0,1


In [14]:
new_date_time_test_df = pd.DataFrame({"month": test_data['pickup_datetime'].dt.month,
              "hour": test_data['pickup_datetime'].dt.hour,
              "week": test_data['pickup_datetime'].dt.isocalendar().week,
              "dayofweek": test_data['pickup_datetime'].dt.dayofweek
             })
# weekday feature
new_date_time_test_df["is_weekday"] = (new_date_time_test_df["dayofweek"]<=4).astype(int)
new_date_time_test_df.head()

,month,hour,week,dayofweek,is_weekday
0,1,17,4,6,0
1,11,11,48,2,1
2,5,2,19,5,0
3,3,18,11,0,1
4,9,1,36,1,1


In [15]:
train_data_new = pd.concat([train_data,new_date_time_train_df],axis=1)
test_data_new = pd.concat([test_data,new_date_time_test_df],axis=1)

In [16]:
# Calculate trip duration in minutes
train_data_new["trip_duration_mins"] = (train_data["dropoff_datetime"]-train_data["pickup_datetime"]).dt.total_seconds()/60
test_data_new["trip_duration_mins"] = (test_data["dropoff_datetime"]-test_data["pickup_datetime"]).dt.total_seconds()/60

In [17]:
# calculate stright line distance between coordinates (Haversine)
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))
train_data_new["straight_line_dist"] = haversine(
    train_data_new['pickup_latitude'], train_data_new['pickup_longitude'],
    train_data_new['dropoff_latitude'], train_data_new['dropoff_longitude']
)
test_data_new["straight_line_dist"] = haversine(
    test_data_new['pickup_latitude'], test_data_new['pickup_longitude'],
    test_data_new['dropoff_latitude'], test_data_new['dropoff_longitude']
)

In [18]:
train_data_new["traffic_congestion_level"].value_counts()

traffic_congestion_level
high      6765
medium    4856
low       3078
Name: count, dtype: int64

In [19]:
# Mpa text to numeric
traffic_map = {
    "low":1,
    "medium":2,
    "high":3
}
train_data_new["traffic_congestion_level"] = train_data_new["traffic_congestion_level"].map(traffic_map)
test_data_new["traffic_congestion_level"] = test_data_new["traffic_congestion_level"].map(traffic_map)

In [20]:
train_data_new.shape,test_data_new.shape

((14699, 21), (300, 21))

In [21]:
train_data_new.head()

,pickup_datetime,dropoff_datetime,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,passenger_count,traffic_congestion_level,pickup_borough,dropoff_borough,...,tip_amount,fare_estimate,trip_distance_miles,month,hour,week,dayofweek,is_weekday,trip_duration_mins,straight_line_dist
0,2024-03-22 09:10:00,2024-03-22 09:23:00,40.690045,-74.239699,40.704018,-74.214368,2,3,Other,Other,...,0.14,18.40,10.9422,3,9,12,4,1,13.0,1.641014
1,2024-08-14 10:48:00,2024-08-14 10:52:00,40.763224,-74.053604,40.759277,-74.066002,1,2,Other,Other,...,1.36,9.99,1.2817,8,10,33,2,1,4.0,0.703825
2,2024-06-17 03:47:00,2024-06-17 03:55:00,40.658970,-74.179122,40.628631,-74.180385,1,1,Other,Staten Island,...,2.36,20.22,13.8034,6,3,25,0,1,8.0,2.097291
3,2024-06-25 00:45:00,2024-06-25 01:04:00,40.696980,-73.773970,40.670444,-73.838194,2,2,Brooklyn,Brooklyn,...,3.80,29.73,24.2266,6,0,26,1,1,19.0,3.832123
4,2024-06-10 04:10:00,2024-06-10 04:14:00,40.580638,-73.788101,40.594286,-73.780625,3,1,Brooklyn,Brooklyn,...,2.68,11.05,3.4222,6,4,24,0,1,4.0,1.021333


In [22]:
train_data_new.columns

Index(['pickup_datetime', 'dropoff_datetime', 'pickup_latitude',
       'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude',
       'passenger_count', 'traffic_congestion_level', 'pickup_borough',
       'dropoff_borough', 'tolls_amount', 'tip_amount', 'fare_estimate',
       'trip_distance_miles', 'month', 'hour', 'week', 'dayofweek',
       'is_weekday', 'trip_duration_mins', 'straight_line_dist'],
      dtype='object')

In [23]:
# Numeric columns to scale
numeric_features = ['straight_line_dist','trip_duration_mins','fare_estimate','tip_amount','tolls_amount',
                   'traffic_congestion_level','passenger_count','hour','month','dayofweek','week','is_weekday']
# Categorical columns to onehot
categorical_features = ["pickup_borough","dropoff_borough"]
# target 
targrt = "trip_distance_miles"
# total
all_features = numeric_features+categorical_features

In [24]:
X = train_data_new[all_features]
y = train_data_new[targrt]
# split data: train, val
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,random_state=42)
X_test = test_data_new[all_features]

In [25]:
# Numeric pipeline
numerical_transformer = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scalar",StandardScaler())
])
categorical_transformer = Pipeline([
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num",numerical_transformer,numeric_features),
        ("cat",categorical_transformer,categorical_features)
    ]
)

# Linear regression

In [27]:
# Linear regression 
lr_model = Pipeline([
    ("preprocessor",preprocessor),
    ("model",LinearRegression())
])

In [28]:
X_train.loc[5191,:]

straight_line_dist          1.688525
trip_duration_mins               7.0
fare_estimate                  12.18
tip_amount                      2.03
tolls_amount                     0.0
traffic_congestion_level           1
passenger_count                    3
hour                               0
month                              1
dayofweek                          2
week                               3
is_weekday                         1
pickup_borough                 Other
dropoff_borough                Other
Name: 5191, dtype: object

In [29]:
lr_model.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [41]:
def evaluate(name, pipeline, X, y):
    preds = pipeline.predict(X)
    mae   = mean_absolute_error(y, preds)
    rmse  = np.sqrt(mean_squared_error(y, preds))
    r2    = r2_score(y, preds)
    print(f"Model:{name}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2: {r2:.4f}")
    print("")

In [31]:
print(evaluate("Linear Regression",lr_model,X_train,y_train))
print(evaluate("Linear Regression",lr_model,X_val,y_val))

Model:Linear Regression
MAE: 3.1423
RMSE: 4.0221
R2: 0.7400

None
Model:Linear Regression
MAE: 3.1743
RMSE: 3.9737
R2: 0.7500

None


In [32]:
-cross_val_score(estimator=lr_model,X,y,scoring='neg_root_mean_squared_error',cv=3,n_jobs=-1)

array([0.69476717, 0.74248025, 0.76970093])

# Random forest

In [64]:
rf_model = Pipeline([
    ("preprocessor",preprocessor),
    ("model",RandomForestRegressor(random_state=42))
])

In [37]:
rf_model.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [38]:
print(evaluate("Random Forest",rf_model,X_train,y_train))
print(evaluate("Random Forest",rf_model,X_val,y_val))

Model:Random Forest
MAE: 0.2324
RMSE: 0.2979
R2: 0.9986

None
Model:Random Forest
MAE: 0.6044
RMSE: 0.7683
R2: 0.9907

None


In [43]:
-cross_val_score(estimator=lr_model,X=X,y=y,scoring='neg_root_mean_squared_error',cv=3,n_jobs=-1)

array([3.94679543, 3.89537011, 4.24717004])

In [44]:
# model overfitting

In [65]:
param_distributions = {
    "model__n_estimators":randint(10,100),
    "model__max_depth":randint(1,20),
    "model__min_samples_leaf":randint(1,50),
    "model__min_samples_split":randint(5,100),
    "model__max_features":uniform(0.7, 0.3)
}

In [66]:
rf_random_search = RandomizedSearchCV(
    rf_model,
    param_distributions,
    n_iter=20,
    scoring = "neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    random_state=42
)

In [67]:
rf_random_search.fit(X_train,y_train)

,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'model__max_depth': <scipy.stats....x71b538bc4670>, 'model__max_features': <scipy.stats....x71b538bc5930>, 'model__min_samples_leaf': <scipy.stats....x71b538bc5a50>, 'model__min_samples_split': <scipy.stats....x71b538bc5c90>, ...}"
,n_iter,10
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [70]:
rf_random_search.best_params_

{'model__max_depth': 10,
 'model__max_features': np.float64(0.7139996989640846),
 'model__min_samples_leaf': 16,
 'model__min_samples_split': 19,
 'model__n_estimators': 71}

In [71]:
rf_random_search.best_score_

np.float64(-0.7764238076211643)

In [ ]:
evaluate("Tuned RF - Val", rf_random_search.best_estimator_, X_val, y_val)

In [72]:
best_rf_model = rf_random_search.best_estimator_
best_rf_model.fit(X,y)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [73]:
-cross_val_score(best_rf_model,X,y,scoring="neg_root_mean_squared_error",cv=3, n_jobs=-1)

array([0.77012135, 0.78203412, 0.77324231])

In [75]:
# Now its look's like good model

In [76]:
# submission 
test_preds = best_rf_model.predict(X_test)
submission_data["trip_distance_miles"] = test_preds
submission_data.to_csv("Submission_RF_tuned.csv", index=False)